# 正式语料库准确度测试复核

## tl;dr

本笔记本读取全量测试脚本生成的结构化证据，复核文件规模、编码与语言识别、40组人工对齐语料、平台金标准断言和查询耗时。所有断言必须通过，才可将数据写入交付版测试报告。

## Context & Methods

- 测试对象：老师提供并已正式入库的语料库，共4个语料集合。
- 计算入口：`scripts/run_formal_corpus_accuracy.py`。
- 结构化输出：`docs/test-evidence/正式语料库准确度测试结果.json`。
- 准确度口径：有明确文件名语言标识的非空文件用于语言识别；40组人工对齐语料用于类型、配对、导入、结构保留和词性覆盖；已建立索引用固定期望值与独立数据库查询交叉验证。
- 重跑全量计算：在项目根目录运行 `backend/.venv/Scripts/python.exe scripts/run_formal_corpus_accuracy.py --source-root <正式语料库目录> --project-root <项目目录> --output-dir docs/test-evidence`。

## Data

In [1]:
import json
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "test-evidence":
    project_root = project_root.parents[1]
result_path = project_root / "docs" / "test-evidence" / "正式语料库准确度测试结果.json"
data = json.loads(result_path.read_text(encoding="utf-8"))
profile = data["corpus_profile"]
gold = profile["gold"]
platform = data["platform_accuracy"]
print(f"证据文件：{result_path}")
print(f"生成时间：{data['generated_at']}")

证据文件：D:\Desktop\CONC\corpus-platform\docs\test-evidence\正式语料库准确度测试结果.json
生成时间：2026-09-07T15:23:35+0800


In [2]:
summary = profile["summary"]
print("文件总数:", summary["total_files"])
print("总字节数:", summary["total_size_bytes"])
print("语料集合:")
for row in profile["collections"]:
    print(f"  {row['name']}: {row['files']} 个文件, {row['bytes']} 字节")
print("类型分布:", summary["type_counts"])
print("编码分布:", profile["encodings"])

文件总数: 4369
总字节数: 52701982
语料集合:
  test-新时代马克思主义中国化经典文献涉法文本汉英平行语料库-汇总20260522: 56 个文件, 1482803 字节
  中文+官译-原始未抽样-马克思主义中国化经典文献汉英平行语料库（unalignment+untagged）: 2346 个文件, 30199443 字节
  外译-原始未抽样-马克思主义中国化经典文献: 1887 个文件, 16418486 字节
  抽样对齐-马克思主义中国化经典文献汉英平行语料库: 80 个文件, 4601250 字节
类型分布: {'paired_raw_zh_en': 1882, 'paired_tagged_zh_en': 136, 'raw_en': 2030, 'raw_zh': 320, 'unknown': 1}
编码分布: {'utf-8-sig': 4312, 'gb18030': 56, 'utf-16': 1}


## Results

In [3]:
metrics = [
    ("可解码率", profile["decode_success_rate"]),
    ("语言识别准确率", profile["language_accuracy"]),
    ("日期格式有效率", profile["date_format_validity"]),
    ("40组类型识别准确率", gold["type_accuracy"]),
    ("40组自动配对准确率", gold["scanner_pair_accuracy"]),
    ("40组导入成功率", gold["import_success_rate"]),
    ("句级对齐保留率", gold["sentence_alignment_retention"]),
    ("段级对齐保留率", gold["paragraph_alignment_retention"]),
    ("中文词性覆盖率", gold["zh_pos_coverage"]),
    ("英文词性覆盖率", gold["en_pos_coverage"]),
    ("平台金标准通过率", platform["pass_rate"]),
]
for name, value in metrics:
    print(f"{name:<20} {value:>8.4f}%")

可解码率                 100.0000%
语言识别准确率              100.0000%
日期格式有效率               99.9764%
40组类型识别准确率           100.0000%
40组自动配对准确率           100.0000%
40组导入成功率             100.0000%
句级对齐保留率               99.9518%
段级对齐保留率               99.9557%
中文词性覆盖率              100.0000%
英文词性覆盖率              100.0000%
平台金标准通过率             100.0000%


In [4]:
print("40组对齐汇总")
print("  人工样本文件:", gold["file_count"])
print("  成对样本:", gold["pair_count"])
print("  句级可配对容量/实际导入:", gold["pairable_sentence_capacity"], gold["imported_sentence_pairs"])
print("  段级可配对容量/实际导入:", gold["pairable_paragraph_capacity"], gold["imported_paragraph_pairs"])
print("  句编号集合完全一致的文件对:", gold["exact_sentence_set_pair_count"], "/ 40")
print("  段编号集合完全一致的文件对:", gold["exact_paragraph_set_pair_count"], "/ 40")
print("  对齐方法分布:", gold["alignment_method_counts"])

40组对齐汇总
  人工样本文件: 80
  成对样本: 40
  句级可配对容量/实际导入: 10377 10372
  段级可配对容量/实际导入: 2258 2257
  句编号集合完全一致的文件对: 37 / 40
  段编号集合完全一致的文件对: 37 / 40
  对齐方法分布: {'provided_structure_id': 12462, 'provided_structure_order': 167}


In [5]:
print("平台金标准:")
by_category = {}
for item in platform["checks"]:
    stats = by_category.setdefault(item["category"], {"通过": 0, "总数": 0})
    stats["总数"] += 1
    stats["通过"] += int(item["passed"])
for category, stats in by_category.items():
    print(f"  {category}: {stats['通过']}/{stats['总数']}")
print("失败项:", [item["name"] for item in platform["checks"] if not item["passed"]])

平台金标准:
  Word: 6/6
  KWIC: 5/5
  File View: 2/2
  Cluster: 2/2
  N gram: 3/3
  Collocate: 3/3
  Plot: 1/1
  Wordcloud: 1/1
  CQP: 1/1
  Parallel: 2/2
  SQL 交叉核对: 20/20
  Keyword: 2/2
失败项: []


In [6]:
print("20次重复查询耗时（毫秒）")
for row in platform["benchmarks"]:
    print(f"  {row['name']}: median={row['median_ms']:.4f}, p95={row['p95_ms']:.4f}, max={row['max_ms']:.4f}")

20次重复查询耗时（毫秒）
  KWIC 中文农民: median=19.9756, p95=21.2453, max=22.6277
  KWIC 英文正则 peasant: median=88.9062, p95=100.4460, max=102.7285
  中文词表: median=8.6177, p95=10.9367, max=11.4618
  英文开放槽 N gram: median=162.1203, p95=188.8458, max=232.1304
  双语平行检索: median=17.9602, p95=20.9016, max=21.1756


## Takeaways

In [7]:
assert summary["total_files"] == sum(row["files"] for row in profile["collections"])
assert summary["total_files"] == sum(summary["type_counts"].values())
assert profile["decode_success_count"] == summary["total_files"]
assert profile["language_gold_count"] == profile["language_match_count"]
assert gold["file_count"] == 80 and gold["pair_count"] == 40
assert gold["type_accuracy"] == 100.0
assert gold["scanner_pair_accuracy"] == 100.0
assert gold["import_success_rate"] == 100.0
assert gold["sentence_alignment_retention"] >= 99.9
assert gold["paragraph_alignment_retention"] >= 99.9
assert gold["zh_pos_coverage"] == 100.0 and gold["en_pos_coverage"] == 100.0
assert platform["failed_count"] == 0 and platform["pass_rate"] == 100.0
print("复核结论：全部一致性断言通过。")

复核结论：全部一致性断言通过。


结论：全量语料可稳定读取；有明确语言标识的非空文件全部识别正确；40组人工对齐语料全部完成自动配对与导入，少量源文件句段编号集合不完全一致时系统按可核验编号保守对齐；48项平台金标准全部符合固定期望值，且词频与检索结果已通过独立数据库查询交叉验证。